# Filtering, Registration, Histogram Matching, and Projection

This notebook demonstrates the optional helper workflow around the unmixing core: intra-stack z-drift correction, time registration, histogram matching, filtering, max-z projection, visualization in napari, and OMIO-based saving.

Author: Fabrizio Musacchio
Date: June 2026

Run the notebook from top to bottom for the packaged example data. To adapt it to your own microscopy data, change the input path and selected channels first, then tune method-specific parameters as described in the Markdown cells.

Several cells open napari viewers. If you run on a headless system, skip those visualization cells and keep the processing and saving cells.


## Imports

Import the package functions used below and locate the repository root. In notebook form, the project root is discovered from the current working directory so the notebook can be opened either from the repository root or from this `notebooks` folder.


In [ ]:
from __future__ import annotations

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "spectral_unmixing").exists() and (candidate / "example_data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the spectral-unmixing project root. "
                       "Start this notebook from inside the repository.")

from spectral_unmixing.io import load_stack_with_omio, write_stack_with_omio
from spectral_unmixing.registration import correct_intra_stack_z_drift, register_stack
from spectral_unmixing.filters import (
    apply_filters,
    match_histograms_across_time,
    max_z_project)

import omio as om

## Input And Output Paths

Choose the input example data and define where results will be written. To use your own data, this is usually the only cell where you need to change paths.


In [ ]:
INPUT_PATH = (PROJECT_ROOT / "example_data" / "Gockel_Nieves_Rivera_2026" / "Gockel_Nieves_Rivera_2026_5D_stack.tif")
INPUT_NAME = INPUT_PATH.stem

OUTPUT_DIR = INPUT_PATH.parent / "registered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"{INPUT_NAME}_registered.tif"

## Load Stack With OMIO

Load the stack through OMIO via the package I/O helper. The returned array is expected in canonical `TZCYX` order, which keeps time, z, channel, y, and x indexing explicit.


In [ ]:
stack, metadata = load_stack_with_omio(INPUT_PATH)
print(f"Loaded stack: {stack.shape}, axes={metadata.get('axes')}")

## Inspect Stack In napari

Display the loaded stack before registration so that motion artifacts and channel content can be inspected visually.


In [ ]:
om.open_in_napari(stack, metadata, "Unregistered |")

## Correct Intra-Stack Z-Drift

Correct slice-to-slice motion inside each 3D time point. The registration is estimated from the selected stable channel and then applied to the full multichannel stack.


In [ ]:
z_corrected_stack = correct_intra_stack_z_drift(
    stack,
    registration_channel=0,  # can also be 1 if channel 1 is the more stable structure
    method="pystackreg",  # "phase_cross_correlation" "pystackreg" 
    reference_mode="neighbor",  # or "full_projection"
    neighbor_window_size=3,  # 3 -> z-1, z, z+1; 5 -> z-2 ... z+2
    pre_median_filter=True,
    post_median_filter=False,
    median_kernel_size=3,
    verbose=True)

print(f"Z-corrected stack: {z_corrected_stack.shape}")
z_corrected_metadata = om.update_metadata_from_image(metadata, z_corrected_stack)
om.open_in_napari(stack, metadata, "Unregistered |")
om.open_in_napari(z_corrected_stack, z_corrected_metadata, "Z-corrected |")

## Register Stack Across Time

Register the time series to the first time point. Registration shifts are estimated from a max-z projection of the selected channel and then applied to the original stack.


In [ ]:
registered_stack = register_stack(
    z_corrected_stack,
    registration_channel=0,
    method="pystackreg", # phase_cross_correlation or pystackreg
    zrange=None,
    pre_median_filter=True,
    post_median_filter=True,
    median_kernel_size=5)
print(f"Registered stack: {registered_stack.shape}")
registered_metadata = om.update_metadata_from_image(metadata, registered_stack)
om.open_in_napari(registered_stack, registered_metadata, "Registered |")

## Histogram Match Across Time

Match time-point intensity distributions to the reference time point. This can reduce slow brightness changes before filtering and projection.


In [ ]:
# Recommendation: do this after registration and before Z projection so that
# geometry is already aligned, but the full 3D time stacks are still available.
matched_registered_stack = match_histograms_across_time(registered_stack, reference_t=0)
print(f"Histogram matched stack: {matched_registered_stack.shape}")
matched_registered_metadata = om.update_metadata_from_image(metadata, matched_registered_stack)
om.open_in_napari(matched_registered_stack, matched_registered_metadata, "Registered + hist matched |")


## Filter Registered Stack

Apply the configured denoising filters to the registered stack while keeping the canonical `TZCYX` structure intact.


In [ ]:
filtered_stack = apply_filters(
    matched_registered_stack,
    filters=["median", "gaussian"],
    median_size=3,
    gaussian_sigma=1.0,
    apply_3d=False)
print(f"Filtered stack: {filtered_stack.shape}")
filtered_metadata = om.update_metadata_from_image(metadata, filtered_stack)
om.open_in_napari(filtered_stack, filtered_metadata, "Filtered |")


## Max-Z-Project

Collapse the z-axis by maximum-intensity projection while preserving time and channel axes. Use `zrange` to restrict the projected z-slices if needed.


In [ ]:
zrange=(0,10) # None or (start_z, end_z) to specify a range of z-slices to project. 
            # If None, the full z-range is projected.
projected_stack = max_z_project(filtered_stack, zrange=zrange)
print(f"Projected stack: {projected_stack.shape}")
projected_metadata = om.update_metadata_from_image(metadata, projected_stack)
# temp_projected_path = OUTPUT_DIR / "ID14135_TP0_d2_unmixed_fixed_alpha_registered_histmatched_projected_tmp.tif"
# temp_projected_saved = write_stack_with_omio(temp_projected_path, projected_stack, metadata)
om.open_in_napari(projected_stack, projected_metadata, "Projected |")


## Filter Projected Stack Again

Apply a second, usually gentler, filter pass after projection. This can improve visualization of projected data but should be tuned conservatively.


In [ ]:
filtered_projected_stack = apply_filters(
    projected_stack,
    filters=["median", "gaussian"],
    median_size=3,
    gaussian_sigma=1.5,
    apply_3d=False)
print(f"Filtered projected stack: {filtered_projected_stack.shape}")
filtered_projected_metadata = om.update_metadata_from_image(metadata, filtered_projected_stack)
om.open_in_napari(filtered_projected_stack, filtered_projected_metadata, "Filtered Projected |")


## Save Filtered Projected Stack With OMIO

Save the processed stack through OMIO. The input file is not overwritten; results are written to the configured output folder.


In [ ]:
filtered_projected_path = OUTPUT_DIR / f"{INPUT_NAME}_registered_histmatched_filtered_projected_z{zrange[0]}_to_{zrange[1]}.tif"
filtered_projected_saved = write_stack_with_omio(filtered_projected_path, filtered_projected_stack, metadata)


## End

The notebook is complete. Saved outputs can be reopened from the output folder or reused in downstream analysis scripts.
